# Session B - ramp against the learned policy

**Pre-registered outcome order.**

- **Primary** - the arm contrast: mean reward from real EEG, ramp minus policy,
  over ABBA-balanced segments, with its bootstrap interval and each arm's dose.
- **Secondary** - the resonance regression: does the peak track arousal, as the
  simulator assumes? Reported whether or not the primary succeeds.

Nothing else here is confirmatory.

**Check the fairness guards before reading the contrast.** Noise degrades only
the closed-loop arm - the scripted ramp ignores the EEG - so on a bad recording
the ramp wins for a trivial reason, and that looks identical to the simulation
result. The cell below reports quality per arm and the policy's command spread
alongside the number.

In [ ]:
import os, sys, json
from pathlib import Path
# Run from the repo root whichever way the notebook was opened, so session
# paths and figure paths mean the same thing.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))
%matplotlib inline
import pandas as pd
from app.protocol import analysis as an, figures as fg, quality as q

KIND = "full"
# Newest matching session. Rehearsals are searched too, for working on the
# analysis before a session has been recorded; they are not published.
ROOTS = [Path("data/sessions"), Path("data/rehearsals")]
candidates = sorted(
    (p for r in ROOTS if r.is_dir() for p in r.glob(f"*_{KIND}*") if p.is_dir()),
    key=lambda p: p.name)

SESSION = candidates[-1] if candidates else None
sess = an.load(SESSION) if SESSION else None
FIG_DIR = Path("artifacts/session_figs") / SESSION.name if SESSION else None
if sess is None:
    print(f"No full session recorded yet.\\n"
          f"Record one with:  python tools/run_session.py --protocol full\\n"
          f"Searched: {[str(r) for r in ROOTS]}")
else:
    print(SESSION)


In [ ]:
if sess is None:
    print("no session loaded - see the first cell")
else:
    m = sess.manifest
    print(f"protocol   {m['protocol']}")
    print(f"status     {m['status']}   duration {m['duration_s']/60:.1f} min   ticks {m['n_ticks']}")
    print(f"montage    {m.get('montage')}   seed {m.get('seed')}   eeg {m.get('eeg_kind')}")
    print(f"commit     {m.get('git_commit')}   dirty {m.get('git_dirty')}")
    print(f"raw        {None if sess.raw is None else sess.raw.shape}")
    if m.get("abort_reason"): print(f"ABORTED    {m['abort_reason']}")

## Block structure as actually run

In [ ]:
if sess is None:
    print("no session loaded - see the first cell")
else:
    rows = []
    for b in sess.manifest.get("blocks", []):
        rows.append({k: b.get(k) for k in ("name","kind","duration_s","eyes","kss_at_end")})
    display(pd.DataFrame(rows))
    sw = sess.manifest.get("sweeps", {})
    if sw:
        display(pd.DataFrame([{"sweep": k, "order": v["order"],
                               "transition_s": round(v["transition_cost_s"],1)}
                              for k, v in sw.items()]))

## 1. Arm contrast - the primary outcome

In [ ]:
if sess is None:
    print("no session loaded - see the first cell")
else:
    summary = an.arm_summary(sess)
    display(summary.round(4))
    
    contrast = an.arm_contrast(sess)
    if not contrast["ok"]:
        print("not available:", contrast["reason"])
    else:
        print(f"ramp - agent : {contrast['ramp_minus_agent']:+.3f}")
        print(f"positions    : {contrast['positions']}   "
              f"balanced={contrast['balanced']}")
        if not contrast["balanced"]:
            print("  ! the arms did not occupy matching positions, so session")
            print("    drift does not cancel. This number is not a contrast.")
    
        print("\n-- what the interval covers --")
        print(f"segments per arm : {contrast['n_segments_per_arm']}")
        print(f"spread between them : "
              f"{ {k: round(v, 3) for k, v in contrast['segment_spread'].items()} }")
        print(f"resampled in blocks of {contrast['bootstrap_block_len']} ticks"
              f"   [{contrast['ci_lo']:+.3f}, {contrast['ci_hi']:+.3f}]")
        if contrast["interval_supports_inference"]:
            lo, hi = contrast["ci_lo"], contrast["ci_hi"]
            print("interval " + ("excludes" if lo * hi > 0 else "includes") + " zero")
        else:
            print("  ! two segments per arm. The bootstrap resamples ticks inside")
            print("    each segment but cannot resample which segments were drawn,")
            print("    and between-segment variation is the term that dominates at")
            print("    arm level. Quote this night descriptively; the interval earns")
            print("    an inferential reading once ABBA and BAAB nights are pooled.")
    
        print("\n-- fairness guards (decided before reading the number) --")
        print(f"quality comparable : {contrast['quality_comparable']}"
              f"   railed gap agent-ramp "
              f"{contrast['railed_gap_agent_minus_ramp']:+.2f} pp")
        print(f"command spread Hz  : {contrast['freq_sd_hz']}")
        print(f"agent controlling  : {contrast['agent_controlling']}"
              f"   (floor {contrast['min_freq_sd_hz']} Hz)")
        if not contrast["quality_comparable"]:
            print("  ! the agent's segments were noisier - it was handicapped by the")
            print("    recording, not beaten. Report descriptively, no directional claim.")
        if not contrast["agent_controlling"]:
            print("  ! the agent barely varied its command - pinned to a rail or")
            print("    emitting a constant, so this compares a ramp with a fixed tone.")
        print("\n  A wide spread is not evidence of control: a policy responding to")
        print("  off-distribution features wanders across the band and passes this.")
        print("  Separating that from a genuine transfer needs the offline replay.")


### Delivered dose

If the agent delivered materially less light, the contrast is partly a dose
effect and must be reported that way.

In [ ]:
if contrast["ok"]:
    d = contrast["mean_dose"]
    print(f"mean dose   ramp {d['ramp']:.4f}   agent {d['agent']:.4f}")
    print(f"agent / ramp = {contrast['dose_ratio_agent_over_ramp']:.2f}")
    if contrast["dose_ratio_agent_over_ramp"] < 0.9:
        print("  -> arms are not dose-matched; report the contrast alongside this")

## 2. Resonance regression - secondary

In [ ]:
if sess is None:
    print("no session loaded - see the first cell")
else:
    reg = an.resonance_regression(sess)
    if not reg["ok"]:
        print("not available:", reg["reason"])
    else:
        print(f"peak vs arousal : {reg['hz_per_arousal']:+.3f} Hz per unit "
              f"(se {reg['hz_per_arousal_se']:.3f})")
        print(f"drift           : {reg['hz_per_minute_drift']:+.4f} Hz/min "
              f"(se {reg['hz_per_minute_drift_se']:.4f})")
        print(f"{reg['n_sweeps']} sweeps, {reg['dof']} dof, "
              f"arousal spanned {reg['arousal_range'][0]:.3f}.."
              f"{reg['arousal_range'][1]:.3f}")
        if not reg["peaks_resolvable"]:
            print("\n  ! at least one sweep's peak was not resolvable at the probe\n"
                  "    spacing used, so the shift this fits is not separable from\n"
                  "    the grid. Do not quote a shift.")


## Overview

Drowsiness, delivered frequency and KSS on one time axis.

In [ ]:
if sess is None:
    print("no session loaded - see the first cell")
else:
    fg.fig_session_overview(sess)

## Entrainment response by sweep

Each point is the response amplitude at one probe frequency, measured only from
samples where the mask actually delivered that frequency.

In [ ]:
if sess is None:
    print("no session loaded - see the first cell")
else:
    fg.fig_response_curves(sess)

## Secondary outcome - fitted peak per sweep

The primary outcome is the arm contrast above. This is the
fallback when the primary is void: it needs only EEG, strobe and a
spectral fit, so it survives a session where the agent could not be
evaluated.


### Does the probe move what it is measuring?

The resonance tracks drowsiness, and each probe drives drowsiness - strobing near
theta raises theta/alpha, near alpha lowers it. In simulation that coupling is
strong enough that the peak crosses its whole range inside one sweep, so the
curve is partly self-inflicted.

A real head should couple far more weakly: an individual's alpha frequency drifts
over tens of minutes, not over the ten seconds a probe lasts. That is an
assumption, so it is measured rather than argued.

In [ ]:
if sess is None:
    print("no session loaded - see the first cell")
else:
    cpl = an.sweep_coupling_report(sess)
    display(cpl.round(3))
    flagged = cpl[cpl["coupled"] == True]
    if len(flagged):
        print(f"\n  ! {len(flagged)} of {len(cpl)} sweeps show probe-driven state change")
        print("    (|r| >= 0.5 between probe frequency and the drowsiness recorded")
        print("    while that probe ran). The response curve is partly self-inflicted;")
        print("    report the peak shift with that stated.")
    else:
        print("\n  no sweep shows meaningful probe-to-state coupling - the sweep")
        print("  behaved as a passive measurement.")

In [ ]:
if sess is None:
    print("no session loaded - see the first cell")
else:
    blocks = an.sweep_blocks(sess)
    tab = an.peak_table(sess, blocks)
    display(tab.round(2))
    
    # The interval is only meaningful if it is wider than the probe grid can
    # resolve. A full-length rehearsal produced intervals of 0.02 Hz that sat
    # 2.9 Hz from the plant's true peak - precision, not accuracy.
    from app.protocol.analysis import bootstrap_peak, response_curve
    bad = []
    for b in blocks:
        try:
            r = bootstrap_peak(response_curve(sess, b))
            if not r.get("resolvable", True):
                bad.append((b, r["hi"] - r["lo"], r["spacing_hz"]))
        except Exception as err:
            print(f"  {b}: {err}")
    if bad:
        print("\n  ! these sweeps cannot resolve a peak - do not quote them:")
        for b, w, sp in bad:
            print(f"    {b}: interval {w:.3f} Hz against a {sp:.2f} Hz probe spacing")
        print("    The fit is biased, not precise. The resonance result is not")
        print("    reportable until the estimator is fixed.")
    else:
        v = tab.set_index("block")["peak_hz"]
        for a, b in zip(blocks, blocks[1:]):
            if pd.notna(v.get(a)) and pd.notna(v.get(b)):
                print(f"  {a} -> {b}: {v[b]-v[a]:+.2f} Hz")

In [ ]:
if sess is None:
    print("no session loaded - see the first cell")
else:
    fg.fig_peak_shift(sess)

## Descent arms

Equal duration by construction, so the two are directly comparable. Arm order is
counterbalanced across sessions - a single session cannot separate arm from
order.

In [ ]:
if sess is None:
    print("no session loaded - see the first cell")
else:
    fg.fig_descent_arms(sess)

## Safety supervisor

Every clamp applied during the session.

In [ ]:
if sess is None:
    print("no session loaded - see the first cell")
else:
    fg.fig_safety_events(sess)

## Post-session notes

In [ ]:
if sess is None:
    print("no session loaded - see the first cell")
else:
    print(sess.notes if (sess.dir/'notes.md').exists() else 'no notes.md')

## Save figures for the thesis

In [ ]:
if sess is None:
    print("no session loaded - see the first cell")
else:
    written = fg.save_all(sess, FIG_DIR, kind="session")
    for p in written: print(p)

## Rehearsal verification - simulation only

Blank on a hardware session, where the plant's resonance is not knowable. On a
full-length simulated rehearsal these cells answer the question the synthetic
tests could not: **does the fit recover the value the simulator actually used**,
and with what bias and interval coverage?

Run this once before the first real night. If coverage is far from 95 %, the
interval quoted in the results figure is not honest and the sweep needs more
dwell or more probes.

In [ ]:
if sess is None:
    print("no session loaded - see the first cell")
else:
    recovery = an.peak_recovery(sess)
    if recovery.empty:
        print("no ground truth in this recording - hardware sessions have none")
    else:
        display(recovery.round(3))
        s = an.recovery_summary(sess)
        print(f"sweeps            : {s['n_sweeps']}")
        print(f"mean error        : {s['mean_error_hz']:+.3f} Hz   "
              f"(|error| {s['mean_abs_error_hz']:.3f})")
        print(f"interval coverage : {s['coverage']:.0%}   "
              f"mean width {s['mean_ci_width_hz']:.2f} Hz")
        print(f"true peak spanned : {s['true_range_hz'][0]:.2f} .. "
              f"{s['true_range_hz'][1]:.2f} Hz")

In [ ]:
if sess is None:
    print("no session loaded - see the first cell")
else:
    drift = an.drift_recovery(sess)
    if not drift["ok"]:
        print("not available:", drift["reason"])
    else:
        print(f"fitted arousal term : {drift['fitted_hz_per_arousal']:+.3f} Hz per unit")
        print(f"fitted drift term   : {drift['fitted_hz_per_minute_drift']:+.4f} Hz/min")
        print(f"true peak span      : {drift['true_peak_span_hz']:.2f} Hz over "
              f"{drift['true_drowsiness_span']:.2f} of drowsiness")
        print()
        print("The simulator applies no time-dependent drift, so a large fitted")
        print("drift term means the regression is absorbing something it should not.")

## Did the agent respond to the EEG at all?

`agent_controlling` only rules out a policy that stopped moving. A policy fed
features outside anything it trained on wanders across the band and looks
identical from the command trace. The difference is visible only against a
counterfactual: re-run the checkpoint on the recorded observations, then on
the same observations with the EEG time-shuffled. If the commands do not
move, the EEG was not driving them.


In [ ]:
if sess is None:
    print("no session loaded - see the first cell")
else:
    CHECKPOINT = sess.manifest.get("checkpoint")
    if not CHECKPOINT:
        print("no checkpoint recorded in the manifest")
    else:
        pr = an.policy_response(sess, Path(CHECKPOINT))
        if not pr["ok"]:
            print("not available:", pr["reason"])
        else:
            print(f"{pr['n_ticks']} agent ticks")
            print(f"command range          {pr['own_command_range']:.3f}")
            print(f"shift when EEG shuffled {pr['command_shift_when_eeg_scrambled']:.3f}")
            print(f"sensitivity            {pr['sensitivity']:.3f} "
                  f"(threshold {pr['threshold']})")
            if pr["responded_to_eeg"]:
                print("\n  -> the commands tracked the EEG: a poor result here is a")
                print("     transfer finding, not a broken pipeline.")
            else:
                print("\n  ! scrambling the EEG barely moved the commands. The policy")
                print("    was not reading the brain, so the arm contrast measures")
                print("    a ramp against an open-loop wanderer. No directional claim.")
